In [1]:
import time
import threading
import warnings
from datetime import datetime
from typing import Dict , Optional
import pandas as pd # type: ignore
warnings.filterwarnings('ignore')

# interactive broker API live

In [2]:
# !pip install ibapi
# !pip install ib_insync

In [15]:
from ibapi.client import EClient # type: ignore
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order
from ibapi.common import BarData
from util.appenv import APPENV # this is for the custom environment variables and configuration


In [16]:
class TradingApp(EClient,EWrapper,APPENV):
    def __init__(self)->None:
        EClient.__init__(self,self)
        APPENV.__init__(self)
        self.data: Dict[int,pd.DataFrame]={}
    def error(self , reqId ,errorTime,errorCode, errorString, advanceOrderReject):
        print(f'reqId: {reqId}, time: {errorTime}, errorCode: {errorCode}, errorString: {errorString}, orderReject: {advanceOrderReject}')
        
    def get_historical_data(self,reqId:int,contract:Contract)->pd.DataFrame:
        self.data[reqId]=pd.DataFrame(columns=['time','high','low','close'])
        self.data[reqId].set_index('time',inplace=True)
        self.reqHistoricalData(
            reqId=reqId,
            contract=contract,
            endDateTime='',
            durationStr='6 D',
            barSizeSetting='1 hour',
            whatToShow='MIDPOINT',
            useRTH=0,
            formatDate=2,
            keepUpToDate=False,
            chartOptions=[],
        )
        time.sleep(4)
        return self.data[reqId]
    def parse_ib_date(value):

        value = str(value)

        # Unix timestamp from IB
        if value.isdigit() and len(value) == 10:
            return pd.to_datetime(int(value), unit='s')

        # IB formatted date
        return pd.to_datetime(value)
    def historicalData(self, reqId:int, bar:BarData)->None:
        df=self.data[reqId]
        # timestamp = self.parse_ib_date(bar.date)
        
        df.loc[bar.date,['high','low','close']]=[bar.high,bar.low,bar.close]
        df=df.astype(float)
        self.data[reqId]=df
 
    @staticmethod
    def get_stock_contract(symbol:str)->Contract:
        contract=Contract()
        contract.symbol=symbol
        contract.secType='STK'
        contract.exchange='SMART'
        contract.currency='USD'
        return contract
        
    
        


# you need to have TWS API and Trader WorkStation

In [17]:

app=TradingApp()


Loaded: /Users/leo/champlain/agentic-ai/acd/ibt/app/config/.env


In [18]:

app.connect(app.host, app.port, app.client_id)

In [22]:
threading.Thread(target=app.run,daemon=True).start()

reqId: -1, time: 1784962557343, errorCode: 2104, errorString: Market data farm connection is OK:usfarm, orderReject: 
reqId: -1, time: 1784962557344, errorCode: 2106, errorString: HMDS data farm connection is OK:ushmds, orderReject: 
reqId: -1, time: 1784962557344, errorCode: 2158, errorString: Sec-def data farm connection is OK:secdefnj, orderReject: 


In [19]:
nvda=TradingApp.get_stock_contract('IONQ')

In [12]:
nvda

4523723760: 0,IONQ,STK,,0.0,,,SMART,,USD,,,False,,combo:

In [22]:
data=app.get_historical_data(10,nvda)
print(len(data))

0


## These are most useful libraries in python for indicators and chart for data . 
### pip install mplfinance for renko and kagi
## pip install pandas_ta for (indicator like MACD,RSI ATR,..) https://github.com/twopirllc/pandas-ta.git
# pip install plotly  for better visualization, like Kagi and Renko